In [2]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor

rng = np.random.default_rng(42)
n = 3000
X = pd.DataFrame({
    "x1": rng.normal(size=n),
    "x2": rng.normal(size=n),
    "bruit": rng.normal(size=n),
})
y = 2 * (X.x1 > 1) + 0.5 * X.x2 + rng.normal(scale=0.5, size=n)

Xtr, Xte = X[:2000], X[2000:]
ytr, yte = y[:2000], y[2000:]
print(Xtr.shape, Xte.shape)

(2000, 3) (1000, 3)


In [3]:
m = LGBMRegressor(n_estimators=200, learning_rate=0.05, num_leaves=31,
                  min_child_samples=20, subsample=0.8, subsample_freq=1,
                  colsample_bytree=0.8, random_state=42, verbose=-1)
m.fit(Xtr, ytr)

print("MSE train :", np.mean((ytr - m.predict(Xtr))**2))
print("MSE test  :", np.mean((yte - m.predict(Xte))**2))

MSE train : 0.1690833209280992
MSE test  : 0.3250346598815604


In [4]:
for nl, ne in [(2, 200), (31, 200), (255, 200), (31, 10), (31, 3000)]:
    m = LGBMRegressor(n_estimators=ne, learning_rate=0.05, num_leaves=nl,
                      min_child_samples=20, subsample=0.8, subsample_freq=1,
                      colsample_bytree=0.8, random_state=42, verbose=-1)
    m.fit(Xtr, ytr)
    train = np.mean((ytr - m.predict(Xtr))**2)
    test = np.mean((yte - m.predict(Xte))**2)
    print(f"num_leaves={nl:3d}  n_estimators={ne:4d}  train={train:.3f}  test={test:.3f}")

num_leaves=  2  n_estimators= 200  train=0.264  test=0.271
num_leaves= 31  n_estimators= 200  train=0.169  test=0.325
num_leaves=255  n_estimators= 200  train=0.135  test=0.348
num_leaves= 31  n_estimators=  10  train=0.745  test=0.838
num_leaves= 31  n_estimators=3000  train=0.016  test=0.442


In [5]:
m = LGBMRegressor(n_estimators=200, learning_rate=0.05, num_leaves=31,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                  random_state=42, verbose=-1, importance_type="gain")
m.fit(Xtr, ytr)
print(dict(zip(X.columns, m.feature_importances_.round(0))))

{'x1': 7052.0, 'x2': 5844.0, 'bruit': 1879.0}


## Observations

- `num_leaves=2` donne le meilleur test (0,271) : la vraie relation est simple (un seuil sur x1 plus une droite sur x2), donc un modèle simple suffit. Train et test sont proches.
- `num_leaves=255` et `n_estimators=3000` : le train devient excellent (0,135 puis 0,016) mais le test se dégrade (0,348 puis 0,442). Le modèle apprend le hasard par cœur : c'est du surapprentissage.
- `n_estimators=10` : trop peu d'arbres, train et test restent mauvais (0,745 et 0,838). C'est du sous-apprentissage.
- Conclusion : ce qui compte est l'erreur de test, pas celle de train.

In [6]:
for nl in (31, 2):
    m = LGBMRegressor(n_estimators=200, learning_rate=0.05, num_leaves=nl,
                      subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                      random_state=42, verbose=-1, importance_type="gain")
    m.fit(Xtr, ytr)
    imp = m.feature_importances_
    print(nl, "part de bruit :", round(imp[2] / imp.sum() * 100, 1), "%")

31 part de bruit : 12.7 %
2 part de bruit : 0.2 %
